# COMP3070 Formulation 2 with C1 to C8

Boolean assignment variables **X[e,t,r]** select one slot and one room for each exam.
**C7 requires all exam facilities. C8 fits exam duration within one slot.**

Choose **Runtime > Run all** with a CPU runtime. This notebook contains the code, tests, all 20 Moodle instances, and eight synthetic extended examples. No Drive mount or Windows Z3 installation is needed.

The original-format files use explicit **baseline** assumptions: no facility requirements, 60-minute exams and 120-minute slots. These are synthetic neutral values, not data supplied by Moodle. The **extended** examples provide complete metadata and exercise C7/C8. Both modes are labelled in the output.

Edit the three source cells below to change the shared reader, independent checker or Boolean solver. Save changes in this notebook so they survive a runtime reset. The Files-panel copies are temporary.

The formulation 1 teammate should use the same **shared_input.py**, metadata and **SHARED_CONTRACT.md**.

In [1]:
%pip install -q z3-solver==5.3.2.0

## Restore the included data and tests

This setup cell restores support files into a dedicated runtime folder. It includes the eight synthetic .txt/.extra.json pairs. The next source cells write the editable code. Running setup again restores the packaged support files.

In [2]:
# @title Restore input files, tests and teammate contract
import base64
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

PACKAGE = """..."""
work = Path('/content/COMP3070_Formulation2_C1_C8')
work.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(base64.b64decode(PACKAGE))) as archive:
    archive.extractall(work)
print('20 Moodle inputs, 8 synthetic example pairs, tests and shared contract restored.')
%cd /content/COMP3070_Formulation2_C1_C8

20 Moodle inputs, 8 synthetic example pairs, tests and shared contract restored.
/content/COMP3070_Formulation2_C1_C8


## Shared input code for both formulations

Each companion JSON file has exam_requirements, room_facilities, exam_durations and slot_durations. List positions use the existing zero-based IDs. Durations are positive integer minutes. Facility names match exactly. The loader validates and normalizes these fields once for either solver.

Without companion data, it supplies labelled neutral baseline values. Incomplete or invalid metadata is an input error. Valid data with no eligible assignment is UNSAT.

In [3]:
%%writefile shared_input.py
"""Shared input contract for both formulations, with no dependency on Z3.

The text format and enrolment order (exam_id, student_id) follow the supplied
Moodle template. The optional JSON format is our proposed group extension.
"""

import json
import re
from pathlib import Path


METADATA_FIELDS = (
    "exam_requirements", "room_facilities", "exam_durations", "slot_durations"
)


class Instance:
    def __init__(self):
        self.number_of_students = 0
        self.number_of_exams = 0
        self.number_of_slots = 0
        self.number_of_rooms = 0
        self.room_capacities = []
        self.exams_to_students = []
        self.exam_requirements = None
        self.room_facilities = None
        self.exam_durations = None
        self.slot_durations = None
        self.metadata_mode = None


def prepare_metadata(instance):
    """Validate and normalize the four fields, or supply neutral baseline data.

    Calling this from either solver also supports the original Instance class.
    Baseline values are synthetic assumptions, not facts supplied by Moodle.
    Feature matching is case-sensitive; names are not silently corrected.
    """
    values = [getattr(instance, name, None) for name in METADATA_FIELDS]
    if all(value is None for value in values):
        values = [
            [[] for _ in range(instance.number_of_exams)],
            [[] for _ in range(instance.number_of_rooms)],
            [60] * instance.number_of_exams,
            [120] * instance.number_of_slots,
        ]
        mode = "baseline"
    elif any(value is None for value in values):
        raise ValueError("C7/C8 metadata must include all four fields.")
    else:
        mode = getattr(instance, "metadata_mode", None) or "extended"

    expected_lengths = (
        instance.number_of_exams, instance.number_of_rooms,
        instance.number_of_exams, instance.number_of_slots,
    )
    normalized = []
    for name, value, count in zip(METADATA_FIELDS, values, expected_lengths):
        if not isinstance(value, (list, tuple)) or len(value) != count:
            raise ValueError(f"{name} must contain exactly {count} entries.")
        if name in ("exam_requirements", "room_facilities"):
            rows = []
            for row in value:
                if not isinstance(row, (list, tuple, set, frozenset)) or any(
                    not isinstance(feature, str) or not feature.strip()
                    or feature != feature.strip() for feature in row
                ):
                    raise ValueError(
                        f"{name} entries must be collections of nonempty feature "
                        "names without surrounding whitespace."
                    )
                rows.append(set(row))
            normalized.append(rows)
        else:
            if any(type(minutes) is not int or minutes <= 0 for minutes in value):
                raise ValueError(f"{name} must contain positive integer minutes.")
            normalized.append(list(value))

    for name, value in zip(METADATA_FIELDS, normalized):
        setattr(instance, name, value)
    instance.metadata_mode = mode
    return instance


def load_metadata(instance, filename):
    """Apply a complete JSON file; invalid input is an error, not UNSAT."""
    with open(filename, encoding="utf-8") as handle:
        metadata = json.load(handle)
    if not isinstance(metadata, dict) or set(metadata) != set(METADATA_FIELDS):
        raise ValueError("Metadata JSON must contain exactly: " + ", ".join(METADATA_FIELDS))
    for name in METADATA_FIELDS:
        setattr(instance, name, metadata[name])
    if any(metadata[name] is None for name in METADATA_FIELDS):
        raise ValueError("Metadata fields cannot be null.")
    instance.metadata_mode = "extended"
    return prepare_metadata(instance)


def read_file(filename, metadata_filename=None, *, baseline=False):
    """Read the original .txt file and, by default, its <stem>.extra.json.

    An explicitly supplied metadata path must exist. baseline=True deliberately
    ignores a companion file and uses the neutral C1-C6-equivalent assumptions.
    """
    if baseline and metadata_filename is not None:
        raise ValueError("Choose either baseline mode or a metadata file.")

    def read_attribute(name):
        line = handle.readline()
        match = re.fullmatch(re.escape(name) + r":\s*(\d+)\s*", line)
        if match:
            return int(match.group(1))
        raise ValueError(f"Could not parse {line!r}; expected {name}.")

    instance = Instance()
    with open(filename, encoding="utf-8") as handle:
        instance.number_of_students = read_attribute("Number of students")
        instance.number_of_exams = read_attribute("Number of exams")
        instance.number_of_slots = read_attribute("Number of slots")
        instance.number_of_rooms = read_attribute("Number of rooms")
        for room in range(instance.number_of_rooms):
            instance.room_capacities.append(read_attribute(f"Room {room} capacity"))
        for line in handle:
            match = re.fullmatch(r"\s*(\d+)\s+(\d+)\s*", line)
            if not match:
                raise ValueError(f"Failed to parse registration: {line!r}")
            instance.exams_to_students.append((int(match[1]), int(match[2])))

    companion = Path(filename).with_suffix(".extra.json")
    if not baseline and metadata_filename is not None:
        return load_metadata(instance, metadata_filename)
    if not baseline and companion.exists():
        return load_metadata(instance, companion)
    return prepare_metadata(instance)

Writing shared_input.py


## Independent timetable checker

This ordinary-Python checker tests actual timetable triples against C1-C8. It uses neither Z3 nor the solver's eligibility calculations. Both formulations can use it to validate their SAT outputs.

In [4]:
%%writefile timetable_check.py
"""Independent, ordinary-Python timetable checks for C1-C8.

No Z3 expressions or solver eligibility helpers are used here.
"""

from itertools import combinations


def validate_timetable(instance, timetable):
    """Raise AssertionError for a violation; return None for a valid timetable."""
    assert timetable is not None, "SAT requires a timetable"
    assert len(timetable) == instance.number_of_exams, "Missing or extra exams"
    assert sorted(e for e, _, _ in timetable) == list(
        range(instance.number_of_exams)
    ), "Every exam must appear exactly once"

    fields = [getattr(instance, name, None) for name in (
        "exam_requirements", "room_facilities", "exam_durations", "slot_durations"
    )]
    has_metadata = any(value is not None for value in fields)
    assert not has_metadata or all(value is not None for value in fields), (
        "Incomplete C7/C8 metadata"
    )

    occupied = set()
    for exam, slot, room in timetable:
        assert 0 <= slot < instance.number_of_slots, "Invalid timeslot"
        assert 0 <= room < instance.number_of_rooms, "Invalid room"
        assert (slot, room) not in occupied, "Room double-booked"
        occupied.add((slot, room))
        students = {s for e, s in instance.exams_to_students if e == exam}
        assert len(students) <= instance.room_capacities[room], "Room too small"
        if has_metadata:
            for feature in instance.exam_requirements[exam]:
                assert feature in instance.room_facilities[room], (
                    f"C7: room {room} lacks {feature!r} for exam {exam}"
                )
            assert instance.exam_durations[exam] <= instance.slot_durations[slot], (
                f"C8: exam {exam} is longer than slot {slot}"
            )

    for student in range(instance.number_of_students):
        exams = {e for e, s in instance.exams_to_students if s == student}
        slots = [slot for exam, slot, _ in timetable if exam in exams]
        assert len(slots) == len(set(slots)), "Student has simultaneous exams"
        assert all(abs(a - b) > 1 for a, b in combinations(slots, 2)), (
            "Student has consecutive exams"
        )

Writing timetable_check.py


## Boolean solver source

C1/C2 select exactly one X[e,t,r] for each exam. C3 limits room occupancy, C4 enforces capacity, C5 prevents adjacent student exams, and C6 prevents simultaneous student exams. C7 and C8 force unsuitable assignments false.

The function solve_boolean(instance) returns a result dictionary without printing. The command-line wrapper prints status, timetable, data mode and measurements. Comments label every rule. Save code changes in this cell.

In [5]:
%%writefile formulation2.py
"""Boolean exam timetabling for the group's C1-C8 constraints.

X[e, t, r] means exam e uses timeslot t and room r. IDs are zero-based.
C7 requires all exam facilities; C8 fits each exam within its single slot.
Derived from the supplied Moodle template; the Boolean model and JSON
extension are additions. See SHARED_CONTRACT.md for the integer teammate.
"""

import argparse
from pathlib import Path
from time import perf_counter

import z3

# Re-export these names to preserve the earlier solver's import interface.
from shared_input import Instance, prepare_metadata, read_file


def solve_boolean(instance):
    """Return status, timetable and measurements without printing.

    timetable is a list of (exam_id, slot_id, room_id) tuples for SAT,
    including an empty list when there are no exams, or None otherwise.
    constraint_count counts top-level Z3 assertions, not internal SAT clauses.
    solve_time_ms measures only solver.check().
    """
    build_start = perf_counter()
    students = instance.number_of_students
    exams = instance.number_of_exams
    slots = instance.number_of_slots
    rooms = instance.number_of_rooms

    if any(type(count) is not int or count < 0
           for count in (students, exams, slots, rooms)):
        raise ValueError("Student, exam, slot and room counts must be non-negative integers.")
    if len(instance.room_capacities) != rooms:
        raise ValueError("Provide exactly one capacity for each room.")
    if any(type(capacity) is not int or capacity < 0
           for capacity in instance.room_capacities):
        raise ValueError("Room capacities must be non-negative integers.")
    prepare_metadata(instance)

    exam_students = [set() for _ in range(exams)]
    student_exams = [set() for _ in range(students)]
    for exam, student in instance.exams_to_students:
        if not (type(exam) is int and type(student) is int
                and 0 <= exam < exams and 0 <= student < students):
            raise ValueError(
                f"Registration ({exam}, {student}) contains an out-of-range ID."
            )
        exam_students[exam].add(student)
        student_exams[student].add(exam)

    solver = z3.Solver()
    X = {
        (exam, slot, room): z3.Bool(f"X_{exam}_{slot}_{room}")
        for exam in range(exams)
        for slot in range(slots)
        for room in range(rooms)
    }

    # C1 and C2: exactly one (timeslot, room) pair for every exam.
    for exam in range(exams):
        choices = [X[exam, slot, room]
                   for slot in range(slots) for room in range(rooms)]
        if choices:
            solver.add(z3.PbEq([(choice, 1) for choice in choices], 1))
        else:
            solver.add(False)

    # C3: a room can host at most one exam in a timeslot.
    if exams > 1:
        for slot in range(slots):
            for room in range(rooms):
                solver.add(z3.AtMost(
                    *[X[exam, slot, room] for exam in range(exams)], 1
                ))

    # C4: rule out assignments to rooms that cannot hold the enrolment.
    for exam in range(exams):
        for room in range(rooms):
            if len(exam_students[exam]) > instance.room_capacities[room]:
                for slot in range(slots):
                    solver.add(z3.Not(X[exam, slot, room]))

    # C7: every required facility must be present. These sets are fixed data;
    # Z3 chooses the assignment, not what facilities the room happens to have.
    for exam in range(exams):
        for room in range(rooms):
            if not instance.exam_requirements[exam].issubset(instance.room_facilities[room]):
                for slot in range(slots):
                    solver.add(z3.Not(X[exam, slot, room]))

    # C8: an exam must fit within one slot. Equality is allowed. C1/C2 still
    # prevent splitting across slots; C3 still reserves the room for the slot.
    for exam in range(exams):
        for slot in range(slots):
            if instance.exam_durations[exam] > instance.slot_durations[slot]:
                for room in range(rooms):
                    solver.add(z3.Not(X[exam, slot, room]))

    # These are Boolean expressions over X, not additional decision variables.
    in_slot = {
        (exam, slot): z3.Or(
            [X[exam, slot, room] for room in range(rooms)]
        )
        for exam in range(exams) for slot in range(slots)
    }
    for registered_exams in student_exams:
        if len(registered_exams) < 2:
            continue
        registered_exams = sorted(registered_exams)

        # C6: the student's exams cannot share a slot, even in different rooms.
        for slot in range(slots):
            solver.add(z3.AtMost(
                *[in_slot[exam, slot] for exam in registered_exams], 1
            ))

        # C5: the student cannot be busy in two adjacent ordered slots.
        busy = [
            z3.Or([in_slot[exam, slot] for exam in registered_exams])
            for slot in range(slots)
        ]
        for first, second in zip(busy, busy[1:]):
            solver.add(z3.Not(z3.And(first, second)))

    build_time_ms = (perf_counter() - build_start) * 1000
    check_start = perf_counter()
    status = solver.check()
    solve_time_ms = (perf_counter() - check_start) * 1000

    timetable = None
    if status == z3.sat:
        model = solver.model()
        timetable = [
            key for key, variable in X.items()
            if z3.is_true(model.eval(variable, model_completion=True))
        ]

    return {
        "status": str(status),
        "timetable": timetable,
        "solve_time_ms": solve_time_ms,
        "build_time_ms": build_time_ms,
        "variable_count": len(X),
        "constraint_count": len(solver.assertions()),
        "reason_unknown": solver.reason_unknown() if status == z3.unknown else None,
        "metadata_mode": instance.metadata_mode,
    }


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("filename", nargs="?", default=Path(__file__).parent / "instances" / "sat1.txt")
    group = parser.add_mutually_exclusive_group()
    group.add_argument("--metadata", help="Explicit companion JSON path")
    group.add_argument("--baseline", action="store_true", help="Ignore metadata and use neutral defaults")
    args = parser.parse_args()
    try:
        result = solve_boolean(read_file(args.filename, args.metadata, baseline=args.baseline))
    except (OSError, ValueError) as error:
        raise SystemExit(f"Input error: {error}") from error
    print(result["status"])
    if result["metadata_mode"] == "baseline":
        print("Data mode: baseline (synthetic neutral defaults; C7/C8 are nonbinding)")
    else:
        print("Data mode: extended (facilities C7 and duration C8)")
    if result["status"] == "sat":
        print("Exam\tTimeslot\tRoom")
        for exam, slot, room in result["timetable"]:
            print(f"{exam}\t{slot}\t{room}")
    elif result["status"] == "unknown":
        print(f'Reason: {result["reason_unknown"]}')
    print(f'Solving time (ms): {result["solve_time_ms"]:.3f}')
    print(f'Model construction time (ms): {result["build_time_ms"]:.3f}')
    print(f'Boolean variables: {result["variable_count"]}')
    print(f'Z3 assertions: {result["constraint_count"]}')

Writing formulation2.py


## Run correctness checks and record measurements

The tests cover 20 original inputs, eight saved extended examples, targeted facility/duration cases, malformed metadata, and 96 tiny cases checked against exhaustive enumeration (48 baseline and 48 extended).

The measurement CSV records **one solve per input**, with data mode, Python/Z3 versions, construction time and solver.check() time. Timings are single-run measurements. Assertion counts mean top-level Z3 assertions. A failed command raises an error.

In [6]:
import platform
import subprocess
import sys
import z3

def run_and_show(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout, end='')
    print(completed.stderr, end='')
    completed.check_returncode()

print('Python:', platform.python_version(), 'Z3:', z3.get_version_string(), flush=True)
run_and_show([sys.executable, '-m', 'unittest', '-v'])
run_and_show([sys.executable, 'test_formulation2.py', '--benchmark'])

Python: 3.13.18 Z3: 5.3.2
test_c7_c8_combination_can_exclude_all_timetables (test_c7_c8.ExtensionTests.test_c7_c8_combination_can_exclude_all_timetables) ... ok
test_c7_does_not_override_capacity (test_c7_c8.ExtensionTests.test_c7_does_not_override_capacity) ... ok
test_c7_empty_requirements_allow_a_room_without_facilities (test_c7_c8.ExtensionTests.test_c7_empty_requirements_allow_a_room_without_facilities) ... ok
test_c7_feature_matching_is_case_sensitive (test_c7_c8.ExtensionTests.test_c7_feature_matching_is_case_sensitive) ... ok
test_c7_missing_facility_is_unsat (test_c7_c8.ExtensionTests.test_c7_missing_facility_is_unsat) ... ok
test_c7_partial_requirements_are_insufficient (test_c7_c8.ExtensionTests.test_c7_partial_requirements_are_insufficient) ... ok
test_c7_requires_every_feature_and_allows_extra_features (test_c7_c8.ExtensionTests.test_c7_requires_every_feature_and_allows_extra_features) ... ok
test_c8_does_not_override_student_gap (test_c7_c8.ExtensionTests.test_c8_does_not

## Run the full model and compare its baseline

The first example is SAT with both facilities and duration active. In the second, both exams need computers and a 90-minute slot. Only one room supplies computers and only one slot is long enough, so C3 makes the combined case UNSAT. Ignoring its metadata restores baseline feasibility.

In [7]:
run_and_show([sys.executable, 'formulation2.py', 'examples/sat_combined_gap.txt'])
print('\nCombined restrictions:')
run_and_show([sys.executable, 'formulation2.py', 'examples/unsat_combined_contention.txt'])
print('\nSame text file with neutral baseline data:')
run_and_show([sys.executable, 'formulation2.py', 'examples/unsat_combined_contention.txt', '--baseline'])

sat
Data mode: extended (facilities C7 and duration C8)
Exam	Timeslot	Room
0	2	0
1	0	0
Solving time (ms): 1.894
Model construction time (ms): 10.435
Boolean variables: 6
Z3 assertions: 11

Combined restrictions:
unsat
Data mode: extended (facilities C7 and duration C8)
Solving time (ms): 0.389
Model construction time (ms): 10.040
Boolean variables: 8
Z3 assertions: 14

Same text file with neutral baseline data:
sat
Data mode: baseline (synthetic neutral defaults; C7/C8 are nonbinding)
Exam	Timeslot	Room
0	0	1
1	0	0
Solving time (ms): 1.226
Model construction time (ms): 7.281
Boolean variables: 8
Z3 assertions: 6


## Edit a personal example

This copies sat2.txt to my_instance.txt and writes its companion metadata. Edit the four arrays below, then rerun this cell. It leaves the test fixtures intact. Every exam must still fit its chosen room, and the shared student needs a gap between exam slots.

In [8]:
import json
from pathlib import Path

Path('my_instance.txt').write_text(Path('instances/sat2.txt').read_text(), encoding='utf-8')
metadata = {
    'exam_requirements': [['computers'], []],
    'room_facilities': [['computers']],
    'exam_durations': [90, 60],
    'slot_durations': [60, 90, 120],
}
Path('my_instance.extra.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
run_and_show([sys.executable, 'formulation2.py', 'my_instance.txt'])

sat
Data mode: extended (facilities C7 and duration C8)
Exam	Timeslot	Room
0	2	0
1	0	0
Solving time (ms): 1.935
Model construction time (ms): 9.913
Boolean variables: 6
Z3 assertions: 11


## Save a package for your teammate

Run this cell to package the current code, data, tests, shared contract and measurement CSV. In the Files panel, open /content/COMP3070_Formulation2_C1_C8 and download COMP3070_Formulation2_C1_C8_Files.zip.

Send **SHARED_CONTRACT.md**, **shared_input.py**, **timetable_check.py** and the example pairs to the formulation 1 teammate. They should use the same facility subset rule and duration <= slot length rule. Compare SAT/UNSAT and schedule validity; the two timetables need not match.

The original parser/Instance design is derived from the supplied Moodle template; the Boolean model, companion metadata and tests are additions. Sources: supplied COMP3070 brief, leader's Symbolic AI.docx, Moodle template and instances, and the [Z3 Python API](https://z3prover.github.io/api/html/namespacez3py.html).

In [12]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

package = Path('COMP3070_Formulation2_C1_C8_Files.zip')
files = sorted(Path('.').glob('*.py')) + sorted(Path('.').glob('*.md'))
files += [Path('requirements.txt'), Path('results_c1_c8.csv')]
files += sorted(Path('instances').glob('*.txt')) + sorted(Path('official').glob('*.py'))
files += sorted(Path('examples').glob('*'))
files += [Path('my_instance.txt'), Path('my_instance.extra.json')]
with ZipFile(package, 'w', ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.as_posix())
print('Ready:', package.resolve())
print('Measurement results: results_c1_c8.csv (baseline and extended modes labelled).')
print('Download the ZIP from the Files panel; save this notebook to preserve source edits.')


Ready: /content/COMP3070_Formulation2_C1_C8/COMP3070_Formulation2_C1_C8_Files.zip
Measurement results: results_c1_c8.csv (baseline and extended modes labelled).
Download the ZIP from the Files panel; save this notebook to preserve source edits.


In [10]:
# Export the measurements produced by this Colab session.
from pathlib import Path
import csv

colab_csv = Path('results_c1_c8.csv').read_text(encoding='utf-8')
colab_rows = list(csv.DictReader(colab_csv.splitlines()))
print('Saved input cases:', len(colab_rows))
print('All expected outcomes match:', all(row['matches_expected'] == 'True' for row in colab_rows))
print('BEGIN_RESULTS_CSV')
print(colab_csv, end='')
print('END_RESULTS_CSV')

Saved input cases: 28
All expected outcomes match: True
BEGIN_RESULTS_CSV
instance,metadata_mode,expected,status,matches_expected,solve_time_ms,build_time_ms,variable_count,constraint_count,python_version,z3_version
instances/sat1.txt,baseline,sat,sat,True,0.6746210000017072,8.98302899997816,1,1,3.13.18,5.3.2
instances/sat10.txt,baseline,sat,sat,True,29.406740000013087,163.71141499999453,300,719,3.13.18,5.3.2
instances/sat2.txt,baseline,sat,sat,True,1.2564110000141682,2.7401530000190633,6,10,3.13.18,5.3.2
instances/sat3.txt,baseline,sat,sat,True,1.045380999983081,1.4063010000029408,4,4,3.13.18,5.3.2
instances/sat4.txt,baseline,sat,sat,True,0.31698999998752697,1.3340819999996256,4,5,3.13.18,5.3.2
instances/sat5.txt,baseline,sat,sat,True,1.2485110000000077,1.318290999989813,4,4,3.13.18,5.3.2
instances/sat6.txt,baseline,sat,sat,True,5.419685000020991,23.206702999999607,90,127,3.13.18,5.3.2
instances/sat7.txt,baseline,sat,sat,True,6.146565999983977,19.750069999986408,100,89,3.13.18,5.3.2
i